### Generate Data by Varying $\omega$

In [ ]:
# Imports
import os, sys
import numpy as np

if "__file__" in globals():
    script_dir = os.path.dirname(os.path.abspath(__file__))
else:
    script_dir = os.getcwd()

utils_dir = os.path.abspath(os.path.join(script_dir, "..", "..", "..", "Utils"))
output_dir = os.path.abspath(os.path.join(script_dir, "..", "Outputs"))

sys.path.append(utils_dir)
from beam_problem import beam_problem as bp
from sobol import generate_sobol
from sensor_processing import sensor_processing
from yaml_processor import load_config, save_config
from data_processing import load_dataset, save_dataset

config_file = os.path.abspath(os.path.join(script_dir, "..", "configuration.yaml"))
config_global = load_config(config_file)

##### Initialization

In [ ]:
# Load or Create New Data
load_init = False
load_train = False
load_val = False
load_test = False

In [ ]:

T_range = np.array([(2/((4**2)*np.pi), 2/np.pi)])  # Time period
w_range = 2*np.pi/(T_range) # Omega
tau_param = np.min(T_range)/5 # Impulse width range
delta_s = 1/(np.sqrt(2)) # Impulse location range
max_dt = tau_param/5 # dt

param_ranges = [w_range[0]]

nt = config_global["data"]["nt"] # number of time steps
nx = config_global["data"]["nx"] # number of spatial points

n_init = 16 # number of initial full space dataset
n_train = 64 # number of initial train params
n_val = 16 # number of validation params per cluster
n_test = 128 # number of test params per cluster 

# Beam Problem
t = np.linspace(0, max_dt * (nt), nt+1)
noise_std = 0.001
beam_problem = bp(nx, nt, i_range = range(1, 250), k_range = range(1, 90), vars=["omega"], upsample=True, t=t, max_dt=max_dt)

##### Generate Initial Full Space Data

In [ ]:
if load_init:
    SS_init, param_init, ft_init, _, _, _ = load_dataset(os.path.join(output_dir, "datasets/Init"), normalize = False)
else:
    param_init = generate_sobol(1, n_init, param_ranges)
    SS_init = np.zeros([n_init, 2, nt, nx])
    ft_init = np.zeros([n_init, nt, len(beam_problem.vars)+1])

    for i in range(n_init):
        print(f"Init - {i+1}/{n_init}")
        SS_init[i, 0], SS_init[i, 1], ft_init[i] = beam_problem.solve(tau = tau_param, s = delta_s, omega = param_init[i])
    
    # save dataset
    save_dataset(os.path.join(output_dir, "datasets/Init"), SS_init, param_init, ft_init, cluster=None)

In [ ]:
import matplotlib.pyplot as plt
plt.figure()
plt.plot(np.linspace(0, 1/np.pi, nt), ft_init[:, :, 0].T)
plt.ylabel("Forcing")
plt.xlabel("Time")
plt.show()

##### Find Optimal Sensor Locations

In [ ]:
# Initialize sensor locator
sp = sensor_processing(SS_init[:, 1], config_global)
sp.perform_svd()
sp.plot_singular(err_cap = 0.00002)

###### Chose number of sensors

In [ ]:
num_sensors = 18

In [ ]:
# Apply sensor locator
sp.opt_sensor_loc(num_sensors, fill_gaps=True)
# Save setup
sp.save(output_dir)

In [ ]:
# Save configuration
config_global["data"]["param_bounds"]["w"] = np.sort(w_range[0]).tolist()
config_global["sensors"]["num_sensors"] = num_sensors
save_config(config_file, config_global)

#### Generate Training Data

In [ ]:
if load_train:
    SS_train, param_train = load_dataset(os.path.join(output_dir, "datasets/Train"), normalize=False)
else:
    param_train = generate_sobol(1, n_train, param_ranges)
    SS_train = np.zeros([n_train, 2, nt, nx])
    ft_train = np.zeros([n_train, nt, len(beam_problem.vars)+1])

    for i in range(n_train):
        print(f"Train - {i+1}/{n_train}")
        SS_train[i, 0], SS_train[i, 1], ft_train[i] = beam_problem.solve(tau = tau_param, s = delta_s, omega = param_train[i])

    # save dataset
    save_dataset(os.path.join(output_dir, "datasets/Train"), SS_train, param_train, ft_train, cluster=None)

#### Generate Validation Data

In [ ]:
if load_val:
    SS_val, param_val = load_dataset(os.path.join(output_dir, "datasets/Val"), normalize=False)
else:
    param_val = generate_sobol(1, n_val, param_ranges)
    SS_val = np.zeros([n_val, 2, nt, nx])
    ft_val = np.zeros([n_val, nt, len(beam_problem.vars)+1])

    for i in range(n_val):
        print(f"Val - {i+1}/{n_val}")
        SS_val[i, 0], SS_val[i, 1], ft_val[i] = beam_problem.solve(tau = tau_param, s = delta_s, omega = param_val[i])

    # save dataset
    save_dataset(os.path.join(output_dir, "datasets/Val"), SS_val, param_val, ft_val, cluster=None)

#### Generate Testing Data

In [ ]:
if load_test:
    SS_test, param_test = load_dataset(os.path.join(output_dir, "datasets/Test"), normalize=False)
else:
    param_test = generate_sobol(1, n_test, param_ranges)
    SS_test = np.zeros([n_test, 2, nt, nx])
    ft_test = np.zeros([n_test, nt, len(beam_problem.vars)+1])

    for i in range(n_test):
        print(f"Test - {i+1}/{n_test}")
        SS_test[i, 0], SS_test[i, 1], ft_test[i] = beam_problem.solve(tau = tau_param, s = delta_s, omega = param_test[i])

    # save dataset
    save_dataset(os.path.join(output_dir, "datasets/Test"), SS_test, param_test, ft_test, cluster=None)

In [ ]:
plt.figure(dpi=200)
plt.plot(np.linspace(0, 1/np.pi, nt)*400*np.pi, ft_test[:, :, 0].T)
plt.ylabel("Forcing")
plt.xlabel("Time")
plt.show()